# Lymph-node stromal and lung fibroblast integration

This notebook contains the study-specific analysis used to compare pulmonary TB
fibroblast states with human lymph-node stromal-cell states.

The input objects are processed and annotated AnnData files generated in the
corresponding scRNA-seq reanalysis workflows. This notebook performs:

1. harmonization of the shared gene space;
2. scVI integration of lymph-node stromal cells and lung fibroblasts;
3. UMAP visualization of the integrated latent representation; and
4. nearest-neighbour enrichment analysis to quantify similarity between lung
   fibroblast subtypes and lymph-node stromal subtypes.

Repository-relative input paths are used in place of local-computer paths.


In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch

from sklearn.neighbors import NearestNeighbors

# RAPIDS was used for neighbour-graph construction and UMAP visualization.
import rapids_singlecell as rsc

np.random.seed(0)
scvi.settings.seed = 0
torch.set_float32_matmul_precision("high")


## Load processed, annotated datasets

Update these repository-relative paths to match the local data layout.

Required metadata:
- `obs["sample"]`
- `obs["sub_type"]`
- `layers["counts"]`


In [ ]:
LN_DATA_PATH = Path("data/adata_LN.h5ad")
LUNG_DATA_PATH = Path("data/adata_Lung.h5ad")

adata_LN = sc.read_h5ad(LN_DATA_PATH)
adata_Lung = sc.read_h5ad(LUNG_DATA_PATH)

required_obs = {"sample", "sub_type"}

for name, adata in {"LN": adata_LN, "Lung": adata_Lung}.items():
    missing_obs = required_obs.difference(adata.obs.columns)
    if missing_obs:
        raise KeyError(f"{name} AnnData is missing obs columns: {sorted(missing_obs)}")
    if "counts" not in adata.layers:
        raise KeyError(f"{name} AnnData is missing layers['counts']")

print("LN:", adata_LN.shape)
print("Lung:", adata_Lung.shape)


## Harmonize genes and construct the integration object

Only genes shared between the two datasets are retained. The count matrices are
used for highly-variable-gene selection and scVI modelling.


In [ ]:
common_genes = adata_Lung.var_names[
    adata_Lung.var_names.isin(adata_LN.var_names)
]

ln_int = adata_LN[:, common_genes].copy()
lung_int = adata_Lung[:, common_genes].copy()

ln_int.obs["tissue"] = "LN"
lung_int.obs["tissue"] = "Lung"

ln_int.obs["study"] = "Grasso"
lung_int.obs["study"] = "Pulmonary_TB"

ln_int.obs["state"] = ln_int.obs["sub_type"].astype(str)
lung_int.obs["state"] = lung_int.obs["sub_type"].astype(str)

ln_int.obs["tissue_state"] = "LN_" + ln_int.obs["state"]
lung_int.obs["tissue_state"] = "Lung_" + lung_int.obs["state"]

# Retain raw counts as the working matrix for concatenation.
ln_int.X = ln_int.layers["counts"].copy()
lung_int.X = lung_int.layers["counts"].copy()

adata_int = ad.concat(
    [ln_int, lung_int],
    join="inner",
    merge="same",
    label="integration_source",
    keys=["LN", "Lung"],
    index_unique="_",
)

adata_int.layers["counts"] = adata_int.X.copy()

print("Shared genes:", len(common_genes))
print(adata_int)


## Highly variable genes

Four thousand highly variable genes are selected using the Seurat v3 method,
accounting for study during HVG selection.


In [ ]:
sc.pp.highly_variable_genes(
    adata_int,
    n_top_genes=4000,
    flavor="seurat_v3",
    layer="counts",
    batch_key="study",
)

adata_scvi = adata_int[:, adata_int.var["highly_variable"]].copy()

print("HVGs:", int(adata_int.var["highly_variable"].sum()))
print(adata_scvi)


## scVI integration

The final analysis used tissue (`LN` versus `Lung`) as the scVI batch covariate.
The model uses a 20-dimensional latent space, two hidden layers and a negative
binomial gene-likelihood model.


In [ ]:
adata_scvi.obs["integration_batch"] = (
    adata_scvi.obs["tissue"]
    .astype(str)
    .map({"LN": "LN_reference", "Lung": "Lung"})
    .astype("category")
)

scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key="integration_batch",
)

model = scvi.model.SCVI(
    adata_scvi,
    n_latent=20,
    n_layers=2,
    n_hidden=128,
    gene_likelihood="nb",
)

model.train(
    max_epochs=400,
    accelerator="auto",
    devices="auto",
    early_stopping=True,
)

adata_scvi.obsm["X_scVI"] = model.get_latent_representation()

print("Latent representation:", adata_scvi.obsm["X_scVI"].shape)


## Integrated UMAP visualization

The UMAP is calculated from the scVI latent representation and is used only for
visualization of the integrated cell states.


In [ ]:
rsc.pp.neighbors(
    adata_scvi,
    use_rep="X_scVI",
    n_neighbors=20,
)

rsc.tl.umap(adata_scvi)

sc.pl.umap(
    adata_scvi,
    color="tissue_state",
    size=20,
    frameon=False,
    legend_loc="right margin",
)


## Lung-to-lymph-node nearest-neighbour analysis

For each lung fibroblast, the 50 nearest lymph-node stromal cells are identified
in the 20-dimensional scVI latent space using Euclidean distance.


In [ ]:
X_scvi = adata_scvi.obsm["X_scVI"]

ln_mask = adata_scvi.obs["tissue"].astype(str).to_numpy() == "LN"
lung_mask = adata_scvi.obs["tissue"].astype(str).to_numpy() == "Lung"

X_ln = X_scvi[ln_mask]
X_lung = X_scvi[lung_mask]

ln_obs = adata_scvi.obs.loc[ln_mask].copy()
lung_obs = adata_scvi.obs.loc[lung_mask].copy()

k = 50

nn_ln = NearestNeighbors(
    n_neighbors=k,
    metric="euclidean",
    n_jobs=-1,
)
nn_ln.fit(X_ln)

_, indices_ln = nn_ln.kneighbors(X_lung)

print("LN cells:", X_ln.shape[0])
print("Lung fibroblasts:", X_lung.shape[0])
print("Nearest neighbours per lung cell:", k)


## Observed-to-expected lymph-node neighbour enrichment

For each lung fibroblast subtype, the observed fraction of its 50 nearest
lymph-node neighbours belonging to each stromal subtype is divided by the
background frequency of that lymph-node subtype. Similarity is reported as
log2(observed / expected).


In [ ]:
ln_subtypes = ln_obs["sub_type"].astype(str).to_numpy()
neighbor_subtypes = ln_subtypes[indices_ln]

ln_states = sorted(np.unique(ln_subtypes))

neighbor_fraction = pd.DataFrame(
    0.0,
    index=lung_obs.index,
    columns=ln_states,
)

for i, cell_id in enumerate(lung_obs.index):
    counts = pd.Series(neighbor_subtypes[i]).value_counts()
    neighbor_fraction.loc[cell_id, counts.index] = counts.to_numpy() / k

neighbor_fraction["lung_subtype"] = (
    lung_obs["sub_type"].astype(str).to_numpy()
)

mean_neighbor_fraction = (
    neighbor_fraction
    .groupby("lung_subtype")[ln_states]
    .mean()
)

ln_background = (
    ln_obs["sub_type"]
    .astype(str)
    .value_counts(normalize=True)
    .reindex(ln_states)
    .astype(float)
)

neighbor_enrichment = mean_neighbor_fraction.div(
    ln_background,
    axis=1,
)

log2_neighbor_enrichment = np.log2(
    neighbor_enrichment + 1e-6
)

log2_neighbor_enrichment


## Plot lung–lymph-node fibroblast similarity

Colour limits are clipped only for visualization; the enrichment values
themselves are not truncated.


In [ ]:
lung_order = [
    "F_ACTA2",
    "F_TIMP1",
    "F_TNXB",
]

ln_order = [
    "CCL19+ SC",
    "CCL21+ SC",
    "CD34+CXCL14+ SC",
    "DES+ SC",
    "GLDN+ SC",
    "HLA-DR+ SC",
    "LAMP5+ SC",
    "NR4A1+BCAM+ SC",
    "Pericytes",
    "SEPT4+ SC",
]

plot_df = log2_neighbor_enrichment.reindex(
    index=lung_order,
    columns=ln_order,
)

vmin, vmax = -2, 2
plot_values = plot_df.clip(lower=vmin, upper=vmax)

fig, ax = plt.subplots(figsize=(11, 4))

im = ax.imshow(
    plot_values.values,
    aspect="auto",
    cmap="RdBu_r",
    vmin=vmin,
    vmax=vmax,
)

ax.set_xticks(np.arange(len(plot_df.columns)))
ax.set_xticklabels(
    plot_df.columns,
    rotation=45,
    ha="right",
)

ax.set_yticks(np.arange(len(plot_df.index)))
ax.set_yticklabels(plot_df.index)

ax.set_xlabel("Lymph-node stromal subtype")
ax.set_ylabel("Lung fibroblast subtype")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log2 observed / expected LN neighbours")

for i in range(plot_df.shape[0]):
    for j in range(plot_df.shape[1]):
        value = plot_df.iloc[i, j]
        if not pd.isna(value):
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

plt.tight_layout()
plt.show()
